# 线性模型（上）线性回归

本节用 PyTorch 实现线性回归，并对比三种求解方式：闭式解、手写梯度下降、`nn.Linear` + 自动微分。

## 生成合成数据

采样一组带高斯噪声的线性关系数据用于演示。

In [ ]:
import torch
import matplotlib.pyplot as plt

torch.manual_seed(0)

N = 200
x = torch.linspace(-3, 3, N).unsqueeze(1)
true_w, true_b = 2.0, 1.0
y = true_w * x + true_b + 0.3 * torch.randn_like(x)

plt.scatter(x.numpy(), y.numpy(), s=8, alpha=0.6)
plt.xlabel('x'); plt.ylabel('y'); plt.title('synthetic data')
plt.show()

## 方法一：最小二乘闭式解

$\hat w = (X^\top X)^{-1} X^\top y$，通常用 `torch.linalg.pinv` 数值稳定地计算。

In [ ]:
# 拼一列常数 1 作为偏置项
X = torch.cat([x, torch.ones_like(x)], dim=1)
w_closed = torch.linalg.pinv(X) @ y
print('closed-form  w =', w_closed[0].item(), ' b =', w_closed[1].item())

## 方法二：手写梯度下降

MSE 损失对 $w$ 的梯度是 $\frac{2}{N} X^\top (Xw - y)$。

In [ ]:
w_gd = torch.zeros(2, 1)
lr = 0.05
for step in range(500):
    pred = X @ w_gd
    grad = (X.T @ (pred - y)) * 2.0 / N
    w_gd = w_gd - lr * grad
print('gradient desc w =', w_gd[0].item(), ' b =', w_gd[1].item())

## 方法三：使用 `nn.Linear` 与自动微分

In [ ]:
import torch.nn as nn
import torch.optim as optim

model = nn.Linear(1, 1)
optimizer = optim.SGD(model.parameters(), lr=0.05)
loss_fn = nn.MSELoss()

for epoch in range(500):
    pred = model(x)
    loss = loss_fn(pred, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print('nn.Linear    w =', model.weight.item(), ' b =', model.bias.item())
print('final MSE   =', loss.item())

## 拟合结果可视化

In [ ]:
with torch.no_grad():
    y_pred = model(x)

plt.scatter(x.numpy(), y.numpy(), s=8, alpha=0.5, label='data')
plt.plot(x.numpy(), y_pred.numpy(), 'r-', label='nn.Linear fit')
plt.xlabel('x'); plt.ylabel('y'); plt.legend(); plt.show()

## 多元线性回归

把输入维数从 1 升到 $d$，`nn.Linear(d, 1)` 即可。下面用 $d=5$ 的合成数据演示。

In [ ]:
d = 5
N2 = 500
torch.manual_seed(1)
X2 = torch.randn(N2, d)
true_w2 = torch.randn(d, 1)
true_b2 = torch.randn(1)
y2 = X2 @ true_w2 + true_b2 + 0.1 * torch.randn(N2, 1)

model2 = nn.Linear(d, 1)
opt2 = optim.Adam(model2.parameters(), lr=0.05)

for epoch in range(500):
    loss = nn.functional.mse_loss(model2(X2), y2)
    opt2.zero_grad(); loss.backward(); opt2.step()

print('learned w:', model2.weight.detach().squeeze().tolist())
print('true    w:', true_w2.squeeze().tolist())
print('final MSE:', loss.item())